# Fig 4 — E4 cache-availability boundary (single panel)

Single panel: % of runs landing Ready vs ClampedToOldest as a function
of filter delay. Hard wall at delay = cache_size = 20s. Annotations
carry the "PTS gap = 0 ms" finding for Ready cells (the dropped 2nd
panel was uninformative because every Ready run measured exactly 0).

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "figures"))

import matplotlib.pyplot as plt
import numpy as np

from _data import e4_decision_counts
from _style import (
    COL_CLAMPED,
    COL_READY,
    COLUMN_WIDTH_IN,
    apply_acm_style,
)

apply_acm_style()

CACHE_SIZE = 20

In [ ]:
counts = e4_decision_counts()  # one row per cell, sorted by delay_s

In [ ]:
fig, ax = plt.subplots(figsize=(COLUMN_WIDTH_IN, 2.4), constrained_layout=True)

x = np.arange(len(counts))
ready_pct = counts["ready"] / counts["n_runs"] * 100
clamped_pct = counts["clamped"] / counts["n_runs"] * 100

# Solid Ready bar (full height when 5/5 land Ready, zero otherwise).
bars_ready = ax.bar(x, ready_pct, color=COL_READY, width=0.65, label="Ready")
# Marker for ClampedToOldest cells — drawn as a stub bar so the legend works.
bars_clamped = ax.bar(x, [3 if c > 0 else 0 for c in counts["clamped"]],
                      color=COL_CLAMPED, width=0.65, alpha=0.95, label="ClampedToOldest")

# Annotate each bar.
for i, row in counts.iterrows():
    if row["ready"] > 0:
        ax.text(i, ready_pct.iloc[i] - 6,
                f"PTS gap = 0 ms\n({row['ready']}/{row['n_runs']} runs)",
                ha="center", va="top", fontsize=6.5, color="white")
    if row["clamped"] > 0:
        ax.text(i, 8, f"clamped\n({row['clamped']}/{row['n_runs']} runs)",
                ha="center", va="bottom", fontsize=6.5, color=COL_CLAMPED, weight="bold")

ax.set_ylim(0, 110)
ax.set_yticks([0, 25, 50, 75, 100])
ax.set_ylabel("% of runs landing Ready")
ax.set_xticks(x)
ax.set_xticklabels(counts["delay_s"].astype(str))
ax.set_xlabel("Filter delay (s)")

# Cache boundary marker.
boundary_x = np.interp(CACHE_SIZE, counts["delay_s"], x)
ax.axvline(boundary_x, color="black", linestyle="--", linewidth=0.8, alpha=0.6)
ax.text(boundary_x, 105, f"cache_size = {CACHE_SIZE}",
        fontsize=6.5, ha="center", va="bottom", style="italic")

ax.legend(loc="upper right", frameon=False, fontsize=7)

In [ ]:
fig.savefig(Path.cwd().parent / "figures" / "fig4_e4_cache_boundary.pdf")
fig.savefig(Path.cwd().parent / "figures" / "fig4_e4_cache_boundary.png", dpi=200)